# 02. Data Cleaning, Normalization & Feature Engineering
**Project:** NovaHome International Market Entry Strategy 2026  
**Engagement Phase:** Phase 03 — Python Data Preparation & Exploratory Analysis  
**Author:** Senior Consulting Analyst & Lead Data Analyst  

### Business & Consulting Purpose
Raw data gathered from disparate secondary sources is rarely in the optimal format for modeling. 
In this notebook, we perform the necessary transformations to prepare the market research data for mathematical scoring, SQL database ingestion, and financial modeling:
1. Standardizing and formatting column headers
2. Normalizing country naming strings
3. Enforcing strict numerical data types (float64, int64, categorical)
4. Feature Engineering:
   * Adding `is_domestic_benchmark` boolean flag to isolate US/Canada from expansion candidates.
   * Calculating `urban_population_m` (absolute urban addressable population).
   * Segmenting countries by `purchasing_power_tier`.
5. Validating processed dataset integrity and exporting to `data/processed/clean_market_research.csv`.


In [1]:
# Step 1: Import libraries and load raw data
import os
import pandas as pd
import numpy as np

RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'market_research.csv')
if not os.path.exists(RAW_DATA_PATH):
    RAW_DATA_PATH = os.path.join('data', 'raw', 'market_research.csv')

df_clean = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset with {df_clean.shape[0]} rows and {df_clean.shape[1]} columns.")


Loaded raw dataset with 10 rows and 14 columns.


### Step 2: Standardizing Column Names & String Formatting
Consulting standard: All database and dataframe column names must use snake_case, containing only lowercase alphanumeric characters and underscores, with no leading or trailing whitespace.


In [2]:
# Standardize column headers: strip whitespace, lowercase
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')
print("Standardized Column Headers:")
print(list(df_clean.columns))

# Normalize country strings: strip accidental whitespace and enforce proper Title Case
df_clean['country'] = df_clean['country'].astype(str).str.strip().str.title()
print(f"
Cleaned Country Names:
{df_clean['country'].tolist()}")


Standardized Column Headers:
['country', 'population_m', 'urban_population_pct', 'disposable_income_usd', 'fitness_participation_pct', 'ecommerce_penetration_pct', 'home_fitness_demand_index', 'market_growth_cagr_pct', 'avg_selling_price_usd', 'import_logistics_cost_usd', 'competitor_intensity_score', 'regulatory_complexity_score', 'digital_ad_cost_index', 'data_status']

Cleaned Country Names:
['United States', 'Canada', 'United Kingdom', 'Germany', 'Netherlands', 'Australia', 'Singapore', 'United Arab Emirates', 'Saudi Arabia', 'India']


### Step 3: Type Casting & Memory Optimization
* Ensure financial values and population metrics are explicitly cast to numeric types (`float64`).
* Cast `data_status` to an efficient categorical type with ordered validation.


In [3]:
# Enforce explicit numeric types
numeric_cols = [
    'population_m', 'urban_population_pct', 'disposable_income_usd',
    'fitness_participation_pct', 'ecommerce_penetration_pct',
    'home_fitness_demand_index', 'market_growth_cagr_pct',
    'avg_selling_price_usd', 'import_logistics_cost_usd',
    'competitor_intensity_score', 'regulatory_complexity_score',
    'digital_ad_cost_index'
]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='raise')

# Cast data_status to category
df_clean['data_status'] = df_clean['data_status'].astype('category')
print("Data types successfully cast and verified:")
print(df_clean.dtypes)


Data types successfully cast and verified:
country                          object
population_m                    float64
urban_population_pct            float64
disposable_income_usd             int64
fitness_participation_pct       float64
ecommerce_penetration_pct       float64
home_fitness_demand_index         int64
market_growth_cagr_pct          float64
avg_selling_price_usd             int64
import_logistics_cost_usd         int64
competitor_intensity_score      float64
regulatory_complexity_score     float64
digital_ad_cost_index           float64
data_status                    category
dtype: object


### Step 4: Strategic Feature Engineering
To accelerate market sizing and financial modeling, we create three strategic derived features:
1. `is_domestic_benchmark`: A boolean indicator (`True`/`False`) isolating the US and Canada so expansion candidate calculations are not skewed by domestic baselines.
2. `urban_population_m`: Absolute urban population in millions, calculated as:
   $$\text{urban\_population\_m} = \text{population\_m} \times \left( \frac{\text{urban\_population\_pct}}{100} \right)$$
3. `income_tier`: Macro segmentation of countries into `Tier 1 (High: >$35k)`, `Tier 2 (Upper-Mid: $20k-$35k)`, and `Tier 3 (Emerging: <$20k)`.


In [4]:
# Feature 1: Domestic Benchmark Flag
domestic_markets = ['United States', 'Canada']
df_clean['is_domestic_benchmark'] = df_clean['country'].isin(domestic_markets)

# Feature 2: Absolute Urban Population in Millions
df_clean['urban_population_m'] = (df_clean['population_m'] * (df_clean['urban_population_pct'] / 100.0)).round(2)

# Feature 3: Income Segmentation Tier
def assign_income_tier(income):
    if income >= 35000:
        return 'Tier 1 (High Income >$35k)'
    elif income >= 20000:
        return 'Tier 2 (Upper-Middle $20k-$35k)'
    else:
        return 'Tier 3 (Emerging <$20k)'

df_clean['income_tier'] = df_clean['disposable_income_usd'].apply(assign_income_tier)

print("Engineered Features Sample Table:")
print(df_clean[['country', 'is_domestic_benchmark', 'urban_population_m', 'income_tier']])


Engineered Features Sample Table:
                    country  is_domestic_benchmark  urban_population_m                       income_tier
0             United States                   True              279.97          Tier 1 (High Income >$35k)
1                    Canada                   True               33.13          Tier 1 (High Income >$35k)
2            United Kingdom                  False               57.63          Tier 1 (High Income >$35k)
3                   Germany                  False               65.66          Tier 1 (High Income >$35k)
4               Netherlands                  False               16.70          Tier 1 (High Income >$35k)
5                 Australia                  False               23.21          Tier 1 (High Income >$35k)
6                 Singapore                  False                6.00          Tier 1 (High Income >$35k)
7      United Arab Emirates                  False                8.31          Tier 1 (High Income >$35k)
8    

### Step 5: Export Processed Dataset & Validation Assertions
We save the cleaned and enriched dataset into `data/processed/clean_market_research.csv` and execute assertions to verify file existence and integrity.


In [5]:
# Define processed output path
PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed', 'clean_market_research.csv')
if not os.path.exists(os.path.dirname(PROCESSED_DATA_PATH)):
    PROCESSED_DATA_PATH = os.path.join('data', 'processed', 'clean_market_research.csv')

# Ensure directory exists
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# Export cleaned data
df_clean.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Cleaned dataset successfully written to: {PROCESSED_DATA_PATH}")

# Validation Assertions
assert os.path.exists(PROCESSED_DATA_PATH), "EXPORT FAILED: Processed file does not exist!"
df_verify = pd.read_csv(PROCESSED_DATA_PATH)
assert len(df_verify) == 10, f"Expected 10 countries, found {len(df_verify)}"
assert 'is_domestic_benchmark' in df_verify.columns, "Missing derived feature: is_domestic_benchmark"
assert 'urban_population_m' in df_verify.columns, "Missing derived feature: urban_population_m"
assert df_verify['urban_population_m'].min() > 0, "Non-positive urban population detected"

print("[PASS] All post-cleaning validation assertions verified successfully.")


Cleaned dataset successfully written to: data/processed/clean_market_research.csv
[PASS] All post-cleaning validation assertions verified successfully.
